Step 1: Project Structure

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving project.zip to project.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip project.zip

Archive:  project.zip
 extracting: project/app.py          
 extracting: project/build_rag.py    
 extracting: project/chatbot.py      
   creating: project/data/
  inflating: project/data/conversations.csv  
 extracting: project/demo.mp4        
   creating: project/outputs/
 extracting: project/outputs/hundred_checkpoints.json  
 extracting: project/outputs/persona.json  
 extracting: project/outputs/topic_checkpoints.json  
 extracting: project/persona_extraction.py  
 extracting: project/preprocessing.py  
 extracting: project/README.md       
 extracting: project/requirements.txt  
 extracting: project/topic_segmentation.py  
   creating: project/vectorstore/


In [ ]:
!ls project

app.py	      data	persona_extraction.py  requirements.txt
build_rag.py  demo.mp4	preprocessing.py       topic_segmentation.py
chatbot.py    outputs	README.md	       vectorstore


Step 2: Install Required Libraries

In [ ]:
!pip install -r project/requirements.txt

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-generativeai gradio pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 26.6 MB/s eta 0:00:00


Step 3: Locate the CSV File

In [ ]:
import os

os.listdir("/content/project/data")

['conversations.csv']

In [ ]:
!ls -R

.:
drive  project	project.zip  sample_data

./drive:
MyDrive

./drive/MyDrive:
 AI_-_Data_Scientist-POTHURAJU_MANJULA.pdf  'manju_resume (1).pdf'
'Colab Notebooks'			     manju_resume.pdf
'Copy of conversations.csv'		     project.zip
 InAmigos				    'resume main (1).pdf'
'manju caste.pdf'			    'resume main.pdf'
'Manjula resume.pdf'			    'Untitled folder'

'./drive/MyDrive/Colab Notebooks':
 conversations.csv	    project.zip       Untitled1.ipynb
'Copy of Untitled1.ipynb'   Untitled0.ipynb   Untitled7.ipynb

./drive/MyDrive/InAmigos:
Donation  Tasks

./drive/MyDrive/InAmigos/Donation:

./drive/MyDrive/InAmigos/Tasks:

'./drive/MyDrive/Untitled folder':
Folder

'./drive/MyDrive/Untitled folder/Folder':

./project:
app.py	      data	persona_extraction.py  requirements.txt
build_rag.py  demo.mp4	preprocessing.py       topic_segmentation.py
chatbot.py    outputs	README.md	       vectorstore

./project/data:
conversations.csv

./project/outputs:
hundred_checkpoints.json  persona.json	topi

Step 4: Load the Dataset

In [ ]:
import os

print("Project CSV size:", os.path.getsize("/content/project/data/conversations.csv"))
print("Drive CSV size:", os.path.getsize("/content/drive/MyDrive/Colab Notebooks/conversations.csv"))

Project CSV size: 12666351
Drive CSV size: 12666351


In [ ]:
import os

print(os.path.getsize("/content/project/data/conversations.csv"))

12666351


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/project/data/conversations.csv",
    header=None
)

print("Shape:", df.shape)
df.head()

Shape: (11001, 1)


,0
0,"User 1: Hi! How are you?\nUser 2: Good, thanks..."
1,User 1: Hey how are you doing?\nUser 2: Doing ...
2,"User 1: Hello there, how are you doing today?\..."
3,User 1: hi there!\nUser 2: Hey! How's your day...
4,User 1: How is your day going?\nUser 2: Great!...


In [ ]:
print(df.columns)

Index([0], dtype='int64')


In [ ]:
print(df.iloc[0, 0])

User 1: Hi! How are you?
User 2: Good, thanks for asking! How about yourself?
User 1: I'm doing pretty well.  I'm excited to be moving to a new city soon!
User 2: Oh that's awesome! What city are you moving to?
User 1: I'm moving to Portland, Oregon.  I'm going to be pursuing my culinary dreams there.
User 2: That sounds amazing! I love Portland.  I'm originally from there.
User 1: Really? That's so cool!  Do you still live there?
User 2: No, I moved away a few years ago.  But I still visit my family there often.
User 1: That's great.  Do you have any favorite places to visit in Portland?
User 2: Yes, I love going to Powell's Books.  It's the largest independent bookstore in the world.
User 1: That sounds amazing!  I've never been to Powell's Books before.  I'm definitely going to have to check it out.
User 2: You definitely should!  It's a really cool place.
User 1: Thanks for the recommendation!  I'm sure I'll love it.
User 2: No problem!  I'm glad I could help.



Step 5: Parse Conversations into Individual Messages

In [ ]:
import re

all_messages = []
global_msg_id = 1

for day_idx, row in df.iterrows():
    conversation = str(row[0])

    # Extract User 1/User 2 messages
    matches = re.findall(
        r'(User\s+\d+):\s*(.*?)(?=(?:User\s+\d+:)|$)',
        conversation,
        re.DOTALL
    )

    for speaker, text in matches:
        text = text.strip()

        if text:
            all_messages.append({
                "day": day_idx,
                "message_id": global_msg_id,
                "speaker": speaker,
                "text": text
            })

            global_msg_id += 1

Step 6: Verify the Parsed Messages

In [ ]:
print("Total messages:", len(all_messages))

for msg in all_messages[:10]:
    print(msg)

Total messages: 191592
{'day': 0, 'message_id': 1, 'speaker': 'User 1', 'text': 'Hi! How are you?'}
{'day': 0, 'message_id': 2, 'speaker': 'User 2', 'text': 'Good, thanks for asking! How about yourself?'}
{'day': 0, 'message_id': 3, 'speaker': 'User 1', 'text': "I'm doing pretty well.  I'm excited to be moving to a new city soon!"}
{'day': 0, 'message_id': 4, 'speaker': 'User 2', 'text': "Oh that's awesome! What city are you moving to?"}
{'day': 0, 'message_id': 5, 'speaker': 'User 1', 'text': "I'm moving to Portland, Oregon.  I'm going to be pursuing my culinary dreams there."}
{'day': 0, 'message_id': 6, 'speaker': 'User 2', 'text': "That sounds amazing! I love Portland.  I'm originally from there."}
{'day': 0, 'message_id': 7, 'speaker': 'User 1', 'text': "Really? That's so cool!  Do you still live there?"}
{'day': 0, 'message_id': 8, 'speaker': 'User 2', 'text': 'No, I moved away a few years ago.  But I still visit my family there often.'}
{'day': 0, 'message_id': 9, 'speaker': 'Us

Step 7: Convert to DataFrame

In [ ]:
messages_df = pd.DataFrame(all_messages)

print(messages_df.shape)
messages_df.head()

(191592, 4)


,day,message_id,speaker,text
0,0,1,User 1,Hi! How are you?
1,0,2,User 2,"Good, thanks for asking! How about yourself?"
2,0,3,User 1,I'm doing pretty well. I'm excited to be movi...
3,0,4,User 2,Oh that's awesome! What city are you moving to?
4,0,5,User 1,"I'm moving to Portland, Oregon. I'm going to ..."


Step 8: Install Sentence Transformers

In [ ]:
!pip install -q sentence-transformers

Step 9: Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


Step 10: IMPORTANT DECISION

In [ ]:
messages_subset = messages_df.iloc[:20000].copy()

In [ ]:
messages_subset = messages_df.copy()

Next Step: Topic Detection (Most Important)

Topic 1 → messages 1–25 → summary

Topic 2 → messages 26–60 → summary

Topic 3 → messages 61–90 → summary

In [ ]:
messages_df = pd.DataFrame(all_messages)

print(messages_df.shape)
messages_df.head()

(191592, 4)


,day,message_id,speaker,text
0,0,1,User 1,Hi! How are you?
1,0,2,User 2,"Good, thanks for asking! How about yourself?"
2,0,3,User 1,I'm doing pretty well. I'm excited to be movi...
3,0,4,User 2,Oh that's awesome! What city are you moving to?
4,0,5,User 1,"I'm moving to Portland, Oregon. I'm going to ..."


Step 11: Create a Working Subset

In [ ]:
messages_df = pd.DataFrame(all_messages)

# Use first 20,000 messages for development
messages_subset = messages_df.iloc[:20000].copy()

print(messages_subset.shape)

(20000, 4)


Step 12: Generate Embeddings in Batches

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = messages_subset["text"].tolist()

embeddings = embedder.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embedding shape: (20000, 384)


Step 13: Topic Change Detection

In [ ]:
WINDOW_SIZE = 10

window_texts = []
window_ranges = []

for i in range(0, len(messages_subset), WINDOW_SIZE):

    chunk = messages_subset.iloc[i:i+WINDOW_SIZE]

    text = " ".join(chunk["text"].tolist())

    window_texts.append(text)

    window_ranges.append((
        chunk.iloc[0]["message_id"],
        chunk.iloc[-1]["message_id"],
        i,
        min(i + WINDOW_SIZE - 1, len(messages_subset)-1)
    ))

print("Total windows:", len(window_texts))

Total windows: 2000


Step 14: Inspect the First Few Topics

In [ ]:
window_embeddings = embedder.encode(
    window_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(window_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(2000, 384)


New Step 15: Topic Detection

In [ ]:
TOPIC_THRESHOLD = 0.45


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

topic_checkpoints = []

current_start = 0
current_embeddings = [window_embeddings[0]]

for i in range(1, len(window_embeddings)):

    centroid = np.mean(current_embeddings, axis=0).reshape(1, -1)

    similarity = cosine_similarity(
        window_embeddings[i].reshape(1, -1),
        centroid
    )[0][0]

    if similarity < TOPIC_THRESHOLD:

        topic_checkpoints.append({
            "start_window": current_start,
            "end_window": i - 1
        })

        current_start = i
        current_embeddings = [window_embeddings[i]]

    else:
        current_embeddings.append(window_embeddings[i])

topic_checkpoints.append({
    "start_window": current_start,
    "end_window": len(window_embeddings) - 1
})

print("Topics detected:", len(topic_checkpoints))

Topics detected: 935


New Step 16: Inspect Topics

In [ ]:
for idx, topic in enumerate(topic_checkpoints[:3]):

    start_w = topic["start_window"]
    end_w = topic["end_window"]

    start_msg = window_ranges[start_w][0]
    end_msg = window_ranges[end_w][1]

    print("\n====================")
    print(f"Topic {idx+1}")
    print(f"Messages {start_msg} to {end_msg}")

    start_idx = window_ranges[start_w][2]
    end_idx = window_ranges[end_w][3]

    print("\nSample messages:")

    for msg in messages_subset.iloc[start_idx:min(start_idx+3, end_idx+1)]["text"]:
        print("-", msg)


Topic 1
Messages 1 to 20

Sample messages:
- Hi! How are you?
- Good, thanks for asking! How about yourself?
- I'm doing pretty well.  I'm excited to be moving to a new city soon!

Topic 2
Messages 21 to 30

Sample messages:
- I'll have to try it sometime! What about your Impala?
- It's a 1964 Impala and it's in pretty good condition, but it needs a new paint job. I'm going to start working on that this weekend.
- That's awesome! I love classic cars.

Topic 3
Messages 31 to 40

Sample messages:
- I'm doing well, thanks for asking. What kind of things do you enjoy doing?
- I enjoy reading, cooking, and spending time with my family.
- Reading is a great way to relax. I love to cook too! What is your favorite meal to make?


Step 17: Configure Gemini

In [ ]:
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai

genai.configure(api_key="YOUR_GEMINI_API_KEY")

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini connected!")

Gemini connected!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Step 18: Create a Topic Summary Function

In [ ]:
def summarize_topic(topic_text):
    prompt = f"""
You are summarizing a segment of a conversation.

Summarize ONLY this topic segment.

Focus on:
- Main subject
- Important facts
- Decisions/events

Keep the summary under 80 words.

Conversation:
{topic_text}
"""

    response = model.generate_content(prompt)

    return response.text.strip()

Step 19: Test on the First Topic

In [ ]:
topic = topic_checkpoints[0]

start_w = topic["start_window"]
end_w = topic["end_window"]

start_idx = window_ranges[start_w][2]
end_idx = window_ranges[end_w][3]

topic_messages = messages_subset.iloc[start_idx:end_idx+1]

topic_text = "\n".join(
    f"{row['speaker']}: {row['text']}"
    for _, row in topic_messages.iterrows()
)

print(topic_text[:2000])

User 1: Hi! How are you?
User 2: Good, thanks for asking! How about yourself?
User 1: I'm doing pretty well.  I'm excited to be moving to a new city soon!
User 2: Oh that's awesome! What city are you moving to?
User 1: I'm moving to Portland, Oregon.  I'm going to be pursuing my culinary dreams there.
User 2: That sounds amazing! I love Portland.  I'm originally from there.
User 1: Really? That's so cool!  Do you still live there?
User 2: No, I moved away a few years ago.  But I still visit my family there often.
User 1: That's great.  Do you have any favorite places to visit in Portland?
User 2: Yes, I love going to Powell's Books.  It's the largest independent bookstore in the world.
User 1: That sounds amazing!  I've never been to Powell's Books before.  I'm definitely going to have to check it out.
User 2: You definitely should!  It's a really cool place.
User 1: Thanks for the recommendation!  I'm sure I'll love it.
User 2: No problem!  I'm glad I could help.
User 1: Hey how are y

Step 20: Generate Topic Summaries

In [ ]:
import google.generativeai as genai

API_KEY = "AQ.Ab8RN6K0-algYc8rmtrVq5XDMxmjm4Ql_0u4qOIhTOJsBH8QvA"

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini configured successfully!")

Gemini configured successfully!


Step 21: Test Gemini on One Topic

In [ ]:
def summarize_topic(topic_text):
    prompt = f"""
You are summarizing a segment of a conversation.

Summarize ONLY this topic segment.

Focus on:
- Main subject discussed
- Important facts/events
- User preferences or decisions

Keep the summary under 80 words.

Conversation:
{topic_text}
"""

    response = model.generate_content(prompt)
    return response.text.strip()

In [ ]:
import google.generativeai as genai

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

Step 24: Generate Topic Summaries (First 5)

Since I have already proved that Gemini works. The only problem is the free-tier quota limit (5 requests/minute). Since our dataset has hundreds of topics, using Gemini for every topic is not practical within 24 hours.
So,I used hybrid approach.

Step 25: Local Summarization Function

In [45]:
def summarize_topic_local(topic_messages):
    texts = topic_messages["text"].tolist()

    # Take the first 2 and last 2 messages
    selected = texts[:2]

    if len(texts) > 4:
        selected += texts[-2:]
    else:
        selected += texts[2:]

    summary = " | ".join(selected)

    return summary[:500]

In [46]:
topic = topic_checkpoints[0]

start_w = topic["start_window"]
end_w = topic["end_window"]

start_idx = window_ranges[start_w][2]
end_idx = window_ranges[end_w][3]

topic_messages = messages_subset.iloc[start_idx:end_idx+1]

print(summarize_topic_local(topic_messages))

Hi! How are you? | Good, thanks for asking! How about yourself? | Nice! I've been meaning to try yoga but I've never really had the time. | It's really great for stress relief and it's a great way to get in shape.


Step 26: Generate ALL Topic Summaries

In [47]:
import json

all_topic_summaries = []

for topic_id, topic in enumerate(topic_checkpoints, start=1):

    start_w = topic["start_window"]
    end_w = topic["end_window"]

    start_idx = window_ranges[start_w][2]
    end_idx = window_ranges[end_w][3]

    topic_messages = messages_subset.iloc[start_idx:end_idx+1]

    summary = summarize_topic_local(topic_messages)

    all_topic_summaries.append({
        "topic_id": topic_id,
        "start_message": int(topic_messages.iloc[0]["message_id"]),
        "end_message": int(topic_messages.iloc[-1]["message_id"]),
        "summary": summary
    })

print("Total topic summaries:", len(all_topic_summaries))

Total topic summaries: 935


Step 27: Save topic_checkpoints.json

In [48]:
with open("/content/project/outputs/topic_checkpoints.json", "w") as f:
    json.dump(all_topic_summaries, f, indent=4)

print("Saved topic_checkpoints.json")

Saved topic_checkpoints.json


Step 28: Create 100-Message Checkpoints

In [49]:
hundred_checkpoints = []

for i in range(0, len(messages_subset), 100):

    chunk = messages_subset.iloc[i:i+100]

    summary = summarize_topic_local(chunk)

    hundred_checkpoints.append({
        "start_message": int(chunk.iloc[0]["message_id"]),
        "end_message": int(chunk.iloc[-1]["message_id"]),
        "summary": summary
    })

print("100-message checkpoints:", len(hundred_checkpoints))

100-message checkpoints: 200


Step 29: Save hundred_checkpoints.json

In [50]:
with open("/content/project/outputs/hundred_checkpoints.json", "w") as f:
    json.dump(hundred_checkpoints, f, indent=4)

print("Saved hundred_checkpoints.json")

Saved hundred_checkpoints.json


In [51]:
print(len(all_topic_summaries))
print(len(hundred_checkpoints))

935
200


Next: Part 2 – Persona Extraction

In [52]:
{
  "habits": [],
  "personal_facts": [],
  "personality_traits": [],
  "communication_style": []
}

{'habits': [],
 'personal_facts': [],
 'personality_traits': [],
 'communication_style': []}

Step 30: Extract Persona Signals

In [53]:
import re
from collections import defaultdict

persona = {
    "habits": set(),
    "personal_facts": set(),
    "personality_traits": set(),
    "communication_style": set()
}

for _, row in messages_subset.iterrows():

    text = row["text"].lower()

    # Personal facts
    if "i'm a" in text or "i am a" in text:
        persona["personal_facts"].add(row["text"])

    if "my parents" in text or "my family" in text:
        persona["personal_facts"].add(row["text"])

    # Habits
    habit_keywords = [
        "i like", "i love", "i enjoy",
        "i usually", "i often",
        "every day", "every morning",
        "i always"
    ]

    if any(k in text for k in habit_keywords):
        persona["habits"].add(row["text"])

    # Personality traits
    if any(word in text for word in [
        "excited", "nervous",
        "happy", "funny",
        "stressed", "emotional"
    ]):
        persona["personality_traits"].add(row["text"])

    # Communication style
    if "!" in row["text"]:
        persona["communication_style"].add(
            "Frequently uses exclamation marks"
        )

    if len(row["text"].split()) <= 5:
        persona["communication_style"].add(
            "Often sends short messages"
        )

Step 31: Convert to JSON Format

In [54]:
persona_json = {
    "habits": list(persona["habits"])[:20],
    "personal_facts": list(persona["personal_facts"])[:20],
    "personality_traits": list(persona["personality_traits"])[:20],
    "communication_style": list(persona["communication_style"])
}

persona_json

{'habits': ["I love to coupon and I like to dye my hair blue. I think it's a bold color that makes a statement.",
  'Me too! I love the way they can transport you to another world.',
  "Oh nice! I love jazz. What's your favorite song?",
  "Thanks! I know it's not the most productive use of my time, but it's something I love to do.",
  'Yeah, it would be. I love sewing and I would love to be able to do it for a living.',
  "That sounds really cool! I'm more into jazz, but I love listening to comedy albums.",
  'I love to play basketball, dress up, and drive.',
  "I love the movie Alien. It's so suspenseful and scary. I love the way it builds up the tension and then the scares just keep coming.",
  "That's a great choice! I love the sauce.",
  'I like a lot of different bands, but some of my favorites are The Killers, The Strokes, and Arctic Monkeys.',
  "That's so cool! I love to quilt too! I also drive a Subaru Outback because it's roomy for my quilting supplies and my mountain bike.",

Step 32: Save Persona JSON

In [55]:
import json

with open("/content/project/outputs/persona.json", "w") as f:
    json.dump(persona_json, f, indent=4)

print("Saved persona.json")

Saved persona.json


Step 33: Rebuild Persona for User 1 Only

In [56]:
persona = {
    "habits": set(),
    "personal_facts": set(),
    "personality_traits": set(),
    "communication_style": set()
}

TARGET_USER = "User 1"

user_messages = messages_subset[
    messages_subset["speaker"] == TARGET_USER
]

for _, row in user_messages.iterrows():

    text = row["text"]
    lower = text.lower()

    # Habits
    if any(k in lower for k in [
        "i like", "i love", "i enjoy",
        "i usually", "i often",
        "every day", "every morning",
        "i always"
    ]):
        persona["habits"].add(text)

    # Personal facts
    if any(k in lower for k in [
        "i'm a", "i am a",
        "i work", "my family",
        "my parents"
    ]):
        persona["personal_facts"].add(text)

    # Personality traits
    if any(k in lower for k in [
        "excited", "nervous",
        "happy", "stressed",
        "worried"
    ]):
        persona["personality_traits"].add(text)

    # Communication style
    if "!" in text:
        persona["communication_style"].add(
            "Frequently uses exclamation marks"
        )

    if len(text.split()) <= 5:
        persona["communication_style"].add(
            "Often sends short messages"
        )

In [58]:
persona_json

{'habits': ["I love to coupon and I like to dye my hair blue. I think it's a bold color that makes a statement.",
  "Thanks! I know it's not the most productive use of my time, but it's something I love to do.",
  'I love to play basketball, dress up, and drive.',
  "That's a great choice! I love the sauce.",
  'I like a lot of different bands, but some of my favorites are The Killers, The Strokes, and Arctic Monkeys.',
  'I love working with kids and helping them learn. I also love seeing the excitement on their faces when they learn something new.',
  "It's definitely an experience! I'm also a big fan of Broadway show tunes. I love singing them in my spare time.",
  'That sounds like a lot of fun! I love going for walks in the park too.',
  'That sounds really cool! I am also into reading, I like to read books about history and music. I have a dog husky.',
  "That sounds awesome! I love to cook as well, but I'm not very good at it.",
  'I like RPGs too! Do you have a favorite?',
  "T

Step 34: Create Final Persona JSON

In [57]:
persona_json = {
    "habits": list(persona["habits"])[:15],
    "personal_facts": list(persona["personal_facts"])[:15],
    "personality_traits": list(persona["personality_traits"])[:15],
    "communication_style": list(persona["communication_style"])
}

persona_json

{'habits': ["I love to coupon and I like to dye my hair blue. I think it's a bold color that makes a statement.",
  "Thanks! I know it's not the most productive use of my time, but it's something I love to do.",
  'I love to play basketball, dress up, and drive.',
  "That's a great choice! I love the sauce.",
  'I like a lot of different bands, but some of my favorites are The Killers, The Strokes, and Arctic Monkeys.',
  'I love working with kids and helping them learn. I also love seeing the excitement on their faces when they learn something new.',
  "It's definitely an experience! I'm also a big fan of Broadway show tunes. I love singing them in my spare time.",
  'That sounds like a lot of fun! I love going for walks in the park too.',
  'That sounds really cool! I am also into reading, I like to read books about history and music. I have a dog husky.',
  "That sounds awesome! I love to cook as well, but I'm not very good at it.",
  'I like RPGs too! Do you have a favorite?',
  "T

In [59]:
from collections import Counter

def top_n(items, n=5):
    return [item for item, _ in Counter(items).most_common(n)]

persona_json = {
    "habits": top_n(persona["habits"], 5),
    "personal_facts": top_n(persona["personal_facts"], 5),
    "personality_traits": top_n(persona["personality_traits"], 5),
    "communication_style": list(persona["communication_style"])
}

persona_json

{'habits': ["I love to coupon and I like to dye my hair blue. I think it's a bold color that makes a statement.",
  "Thanks! I know it's not the most productive use of my time, but it's something I love to do.",
  'I love to play basketball, dress up, and drive.',
  "That's a great choice! I love the sauce.",
  'I like a lot of different bands, but some of my favorites are The Killers, The Strokes, and Arctic Monkeys.'],
 'personal_facts': ["It's been pretty good. I'm about to go on a hike. What about you?",
  "I'm an environmental engineer. I love my job because it allows me to have a positive impact on the environment.",
  "I'm doing well, just got home from work. I'm a teacher too.",
  "that's awesome! I'm a writer.",
  "I'm a party planner. I love planning events and making people happy."],
 'personality_traits': ['I like to listen to a variety of music. I love music that makes me feel happy and relaxed. I also love music that makes me feel nostalgic.',
  "Thanks! She is. I'm reall

In [ ]:
with open("/content/project/outputs/persona.json", "w") as f:
    json.dump(persona_json, f, indent=4)

print("persona.json saved")

In [60]:
import os

print(os.listdir("/content/project/outputs"))

['topic_checkpoints.json', 'hundred_checkpoints.json', 'persona.json']


PART 3: Build the RAG Chatbot

Step 36: Install Required Libraries

In [61]:
!pip install -q faiss-cpu sentence-transformers gradio

Step 37: Load Saved Files

In [62]:
import json

with open("/content/project/outputs/topic_checkpoints.json") as f:
    topic_data = json.load(f)

with open("/content/project/outputs/persona.json") as f:
    persona_data = json.load(f)

print("Topics:", len(topic_data))
print("Persona loaded.")

Topics: 935
Persona loaded.


Step 38: Create Retrieval Documents

In [63]:
documents = []

for topic in topic_data:
    documents.append(
        f"Topic {topic['topic_id']} "
        f"(Messages {topic['start_message']}-{topic['end_message']}): "
        f"{topic['summary']}"
    )

print("Documents:", len(documents))
print(documents[0])

Documents: 935
Topic 1 (Messages 1-20): Hi! How are you? | Good, thanks for asking! How about yourself? | Nice! I've been meaning to try yoga but I've never really had the time. | It's really great for stress relief and it's a great way to get in shape.


Step 39: Create Embeddings

In [64]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

doc_embeddings = embed_model.encode(
    documents,
    show_progress_bar=True
)

print(doc_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

(935, 384)


Step 40: Build FAISS Index

In [65]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(doc_embeddings).astype("float32")
)

print("FAISS index size:", index.ntotal)

FAISS index size: 935


Step 41: Create Retrieval Function

In [66]:
def retrieve(query, k=3):

    query_embedding = embed_model.encode(
        [query]
    )

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

In [67]:
retrieve("What hobbies were discussed?")

["Topic 460 (Messages 8711-8730): What are some of your other hobbies? | I like to read, write, and play video games. | They can be at times, but they're also a lot of fun. | I bet they are.",
 "Topic 209 (Messages 4241-4250): I might just do that! | Awesome! I'm sure you'd be great at it. | So, what are your hobbies? | My hobbies include exercising, reading, and spending time with my family.",
 "Topic 180 (Messages 3641-3650): That's awesome. | So, what are your hobbies? | Great! I'm sure you'll be great at it. | Thanks"]

Step 42: Build Chatbot Logic

In [68]:
def chatbot(query):

    q = query.lower()

    if "habit" in q:
        return "\n• ".join(
            ["Observed habits:"] +
            persona_data["habits"]
        )

    elif "how do they talk" in q \
         or "communication" in q:

        return "\n• ".join(
            ["Communication style:"] +
            persona_data["communication_style"]
        )

    elif "what kind of person" in q \
         or "personality" in q:

        return "\n• ".join(
            ["Observed traits:"] +
            persona_data["personality_traits"]
        )

    else:

        retrieved = retrieve(query)

        return (
            "Relevant conversation topics:\n\n"
            + "\n\n".join(retrieved)
        )

Step 43: Test the Chatbot

In [69]:
print(chatbot(
    "What are their habits?"
))

Observed habits:
• I love to coupon and I like to dye my hair blue. I think it's a bold color that makes a statement.
• Me too! I love the way they can transport you to another world.
• Oh nice! I love jazz. What's your favorite song?
• Thanks! I know it's not the most productive use of my time, but it's something I love to do.
• Yeah, it would be. I love sewing and I would love to be able to do it for a living.
• That sounds really cool! I'm more into jazz, but I love listening to comedy albums.
• I love to play basketball, dress up, and drive.
• I love the movie Alien. It's so suspenseful and scary. I love the way it builds up the tension and then the scares just keep coming.
• That's a great choice! I love the sauce.
• I like a lot of different bands, but some of my favorites are The Killers, The Strokes, and Arctic Monkeys.
• That's so cool! I love to quilt too! I also drive a Subaru Outback because it's roomy for my quilting supplies and my mountain bike.
• I love to read children

In [70]:
print(chatbot(
    "How do they talk?"
))

Communication style:
• Often sends short messages
• Frequently uses exclamation marks


In [71]:
print(chatbot(
    "Tell me about hobbies."
))

Relevant conversation topics:

Topic 460 (Messages 8711-8730): What are some of your other hobbies? | I like to read, write, and play video games. | They can be at times, but they're also a lot of fun. | I bet they are.

Topic 209 (Messages 4241-4250): I might just do that! | Awesome! I'm sure you'd be great at it. | So, what are your hobbies? | My hobbies include exercising, reading, and spending time with my family.

Topic 681 (Messages 13251-13260): I like to read, go for walks, and spend time with my dog. | Sounds like you have a lot of hobbies! | Me too! I think it's important to have a lot of hobbies. | I agree! It's fun to have different things to do.


Step 44: Create Gradio Interface

In [72]:
import gradio as gr

demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="Persona + RAG Chatbot",
    description=(
        "Ask about the user's habits, "
        "personality, communication style, "
        "or conversation topics."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76b4ab774a5cd5fee9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
